In [1]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Import all necessary libraries
import pandas as pd 
import re

In [ ]:
# import data 
df_a = pd.read_csv("Part A with Supplier.csv")
df_b = pd.read_csv("Part B with Supplier.csv")

C:\Users\ITafr\AppData\Local\Temp\ipykernel_30140\1202115446.py:2: DtypeWarning: Columns (0,3,4,5,6,7,8,9,10,11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_a = pd.read_csv("Part A with Supplier.csv")
C:\Users\ITafr\AppData\Local\Temp\ipykernel_30140\1202115446.py:3: DtypeWarning: Columns (0,3,4,5,6,7,8,9,10,11,12,13,14,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_b = pd.read_csv("Part B with Supplier.csv")


In [ ]:
frames = [df_a, df_b]
df = pd.concat(frames, ignore_index=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1376677 entries, 0 to 1376676
Data columns (total 29 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   PART_NO           1376677 non-null  object 
 1   DESCRIPTION       1376677 non-null  object 
 2   MFR               1203435 non-null  object 
 3   STYLE             428364 non-null   object 
 4   COMPOSITION       372517 non-null   object 
 5   WIDTH             112200 non-null   object 
 6   COLOR             616087 non-null   object 
 7   TYPE              643611 non-null   object 
 8   SIZE              540604 non-null   object 
 9   GAUGE             335557 non-null   object 
 10  FACE WEIGHT       168144 non-null   object 
 11  COLLECTION        136996 non-null   object 
 12  DEPTH             31424 non-null    object 
 13  BACKING           281935 non-null   object 
 14  FINISH            142852 non-null   object 
 15  SITE              1376677 non-null  object 
 16  

In [ ]:
# How many null entries in part number
num_null_part_numbers = df['PART_NO'].isnull().sum()
print(f"Number of null entries in PART_NO: {num_null_part_numbers}")

Number of null entries in PART_NO: 0


In [ ]:
# Unique descriptions
num_unique_descriptions = df['DESCRIPTION'].nunique()
print(f"Number of unique descriptions: {num_unique_descriptions}")

Number of unique descriptions: 399469


In [ ]:
# Unique parts
num_unique_part = df['PART_NO'].nunique()
print(f"Number of unique Part numbers: {num_unique_descriptions}")

Number of unique Part numbers: 399469


Observation: <br>
    1) There are 398 609 unique parts <br>
    2) all part descriptions have unique part numbers 

In [ ]:
# Return a DataFrame with unique descriptions
unique_df = df.drop_duplicates(subset=['DESCRIPTION'])
unique_df = unique_df.reset_index(drop=True)
unique_df.head(2)

,PART_NO,DESCRIPTION,MFR,STYLE,COMPOSITION,WIDTH,COLOR,TYPE,SIZE,GAUGE,...,COMMODITY 2,PRODUCT CODE,PRODUCT_FAMILY,INV UoM,PURCH UoM,CONV FACTOR,PRIMARY SUPPLIER,SUPPLIER NAME,NET WEIGHT,NET VOLUME
0,1000042658,Miscellaneous Material - CTN,OTHR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2MISC,*,*,CTN,CTN,1.0,999999,SUPPLIER PLACEHOLDER,NaN,NaN
1,1000017103,Vortex 400g Orange,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2VCT,*,*,EA,EA,1.0,102549,"Aramsco, Inc.",NaN,NaN


In [ ]:
# Miscellaneous descriptions
num_miscellaneous_descriptions = unique_df['DESCRIPTION'].str.contains('Miscellaneous', case=False, na=False).sum()
print(f"Number of descriptions with 'Miscellaneous': {num_miscellaneous_descriptions}")

Number of descriptions with 'Miscellaneous': 23


In [ ]:
# Create a boolean mask for PART_NO in unique_df_clean that are also in df_miscellaneous
df_miscellaneous = unique_df[unique_df['DESCRIPTION'].str.contains('Miscellaneous', case=False, na=False)]
unique_df['miscellaneous'] = unique_df['PART_NO'].isin(df_miscellaneous['PART_NO'])

In [ ]:
# Filter out rows where DESCRIPTION starts with "Material -" (case-sensitive) ---
df_material = unique_df[unique_df['DESCRIPTION'].str.startswith("Material -", na=False)]
df_material.head(2)

,PART_NO,DESCRIPTION,MFR,STYLE,COMPOSITION,WIDTH,COLOR,TYPE,SIZE,GAUGE,...,PRODUCT CODE,PRODUCT_FAMILY,INV UoM,PURCH UoM,CONV FACTOR,PRIMARY SUPPLIER,SUPPLIER NAME,NET WEIGHT,NET VOLUME,miscellaneous
9,1000002136,Material - Non-Carpet Glued,OTHR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,EA,EA,1.0,1070,"Noble Flooring, LLC",NaN,NaN,False
10,1000002140,Material - Sundry Carpet Glued Related,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,EA,EA,1.0,SUB,Sub contractor Place holder,NaN,NaN,False


In [ ]:
# Create a boolean mask for PART_NO in unique_df_clean that are also in df_material
unique_df['material_part'] = unique_df['PART_NO'].isin(df_material['PART_NO'])

In [ ]:
# Lets get the manufacture extraction function (extract the first 1 to 3 capitalized words)
def extract_manufacture(description):
    try:
        # Match the first word (letters only), regardless of case
        match = re.match(r'^\s*([a-zA-Z]+)', str(description))
        return match.group(1).strip() if match else None
    except:
        return None

In [ ]:
# Apply supplier extraction to the DESCRIPTION column
unique_df['Manufacture'] = unique_df['DESCRIPTION'].apply(extract_manufacture)
print(unique_df['Manufacture'].nunique())


7475


In [ ]:
# # --- Filter out rows where DESCRIPTION starts with "Material" (case-sensitive) ---
# unique_df_clean = df[~df['DESCRIPTION'].str.startswith("Material", na=False)]

### Lets bite the pie one manufacure at a time <br> 
1) First we need to separate this big data by manufacture. <br> 
2) We need to filter by commodity. <br> 
3) We need to create a functions to extract all the relevant information. 

In [ ]:
df_interface = unique_df[unique_df['DESCRIPTION'].str.contains(r'^interface', case=False, na=False, regex=True)]
df_interface.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12193 entries, 278 to 399465
Data columns (total 32 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PART_NO           12193 non-null  object 
 1   DESCRIPTION       12193 non-null  object 
 2   MFR               11710 non-null  object 
 3   STYLE             8665 non-null   object 
 4   COMPOSITION       530 non-null    object 
 5   WIDTH             554 non-null    object 
 6   COLOR             8645 non-null   object 
 7   TYPE              8671 non-null   object 
 8   SIZE              8174 non-null   object 
 9   GAUGE             832 non-null    object 
 10  FACE WEIGHT       8299 non-null   object 
 11  COLLECTION        0 non-null      object 
 12  DEPTH             0 non-null      object 
 13  BACKING           8347 non-null   object 
 14  FINISH            0 non-null      object 
 15  SITE              12193 non-null  object 
 16  PART STATUS       12193 non-null  object 


In [ ]:
# create a function to extract the relevant information from the description

# Regex for standard format

def extract_description_flexible(desc):
    if not isinstance(desc, str):
        return {}

    desc = desc.strip()

    # Pattern 1: Parentheses-style detailed descriptions
    pattern_1 = re.compile(r"""^Interface(?:flor)?\s+
                                (?P<name>[^\(]+?)\s*
                                (?:\((?P<part_number>[\w\-]+)\))?\s+
                                (?P<product_type>\w+)\s+
                                (?P<form>(?:\w+\s*)+?)
                                (?P<dimensions>[\d\.]+\s*\w+\s+x\s+[\d\.]+\s*\w+)\s*
                                [-–]\s*
                                (?P<color>.+?)\s*
                                \((?P<color_code>[^)]+)\)
                                $""", re.VERBOSE | re.IGNORECASE)

    match_1 = pattern_1.match(desc)
    if match_1:
        return match_1.groupdict()

    # Pattern 2: Backslash-delimited format
    if '\\' in desc:
        parts = desc.split('\\')

        # If there are 4 string component, we assume the last part contains dimensions 
        # INTERFACE\4716 BROADLEAF LOOP\102005 STEPPE\50CMX50CM
        if len(parts) == 4:
            return {
                'collection': parts[1].strip(),                 # Treat 4716 BROADLEAF LOOP as collection
                'product_name': parts[2].strip(),               # 102005 STEPPE
                'backing': None,                                # No backing in this format
                'dimensions': parts[3].strip(),                 # 50CMX50CM
                'name': None,
                'part_number': None,
                'product_type': None,
                'form': None,
                'color': None,
                'color_code': None
            }
        # If there are 5 parts, we assume the last part contains color information  
        # INTERFACE\FOLIO II\9654 CHAMPAGNE\GLASBAC\50CMX50CM
        elif len(parts) == 5:
            return {
                'collection': parts[1].strip(),
                'product_name': parts[2].strip(),
                'backing': parts[3].strip(),
                'dimensions': parts[4].strip(),
                'name': None,
                'part_number': None,
                'product_type': None,
                'form': None,
                'color': None,
                'color_code': None
            }
        
        # If there are 6 parts, we assume the last part contains color information 
        # INTERFACE\CARPET TILE\WORLD WOVEN COLLECTION WW880\PRODUCT #:128200AK00\GlasBac\COLOR- 105364 NATURAL LOOM
        elif len(parts) == 6:
            # Clean up and extract color code and name
            color_raw = parts[5].strip()  # "COLOR- 105364 NATURAL LOOM"
            color_code_match = re.match(r'COLOR-\s*(\d+)\s+(.+)', color_raw, re.IGNORECASE)

            if color_code_match:
                color_code = color_code_match.group(1)  # '105364'
                color_name = color_code_match.group(2)  # 'NATURAL LOOM'
            else:
                color_code = None
                color_name = color_raw  # fallback

            return {
                'category': parts[1].strip(),                     # 'CARPET TILE'
                'collection': parts[2].strip(),                   # 'WORLD WOVEN COLLECTION WW880'
                'product_number': parts[3].replace('PRODUCT #:', '').strip(),  # '128200AK00'
                'backing': parts[4].strip(),                      # 'GlasBac'
                'color': color_name,                              # 'NATURAL LOOM'
                'color_code': color_code,                         # '105364'
                'name': None,
                'part_number': None,
                'product_type': None,
                'form': None,
                'dimensions': None,
                'product_name': None,
            }

    # Pattern 3: No part number, but dash/en dash + color and color code
    pattern_3 = re.compile(r"""^Interface\s+
                                (?P<name>\w+)\s+
                                (?P<product_type>\w+)\s+
                                (?P<form>(?:\w+\s*)+?)
                                (?P<dimensions>[\d\.]+\s*\w+\s+x\s+[\d\.]+\s*\w+)\s*
                                [-–]\s*
                                (?P<color>.+?)\s*
                                \((?P<color_code>[^)]+)\)
                                $""", re.VERBOSE | re.IGNORECASE)

    match_3 = pattern_3.match(desc)
    if match_3:
        return match_3.groupdict()
    # Pattern 4: No dimensions, but color and color code
    pattern_4 = re.compile(r"""^Interface-(?P<collection>[^-]+)\s*-\s*
                            (?P<product_name>[^\(]+?)\s*
                            \((?P<part_number>[\w\-]+)\)-       # e.g., (131160AK0G)
                            (?P<backing>[^\-]+)-                # e.g., CQuestGB
                            (?P<dimensions>[\d\.]+\w+\s*x\s*[\d\.]+\w+)-   # e.g., 9.845in x 39.38in
                            (?P<color>.+?)\s*
                            \((?P<color_code>[^)]+)\)
                            $""", re.VERBOSE | re.IGNORECASE)

    # Fallback if no match
    return {
        'collection': None,
        'product_name': None,
        'backing': None,
        'dimensions': None,
        'name': None,
        'part_number': None,
        'product_type': None,
        'form': None,
        'color': None,
        'color_code': None
    }

In [ ]:
df_sample = df_interface.sample(50, random_state=56)
df_sample.tail(10)

,PART_NO,DESCRIPTION,MFR,STYLE,COMPOSITION,WIDTH,COLOR,TYPE,SIZE,GAUGE,...,INV UoM,PURCH UoM,CONV FACTOR,PRIMARY SUPPLIER,SUPPLIER NAME,NET WEIGHT,NET VOLUME,miscellaneous,material_part,Manufacture
75455,030079,INTERFACE\CLOUD COVER\105707 MEADOW\50CMX50CM,INTF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,CTN,CTN,1.0,999999,SUPPLIER PLACEHOLDER,NaN,NaN,False,False,INTERFACE
303774,184081-015,Interface S201 Glasbac RE (142450250H) Tufted ...,INTF,5355 - S201 Glasbac RE,NaN,NaN,78712 - Sage (9798),2014 - Tufted Patterned Loop,494 - 19.69in x 19.69in,NaN,...,SY,NaN,NaN,NaN,NaN,NaN,NaN,False,False,Interface
319780,171788-023,Interface Finsbury Square GlasBac (1416102500)...,INTF,8025 - Finsbury Square GlasBac,NaN,NaN,10360 - Warwick (102768),2021 - Tufted Sheared,494 - 19.69in x 19.69in,NaN,...,SY,SY,1.0,102775.0,"Interface Americas, Inc.",NaN,NaN,False,False,Interface
265723,192524-001,Interface-SL920 (Silver Linings SL920)-Unbacke...,INTF,10565 - SL920,356 - Undefined,78 - 6',38314 - Beige Line (104519),2029 - Tufted Textured Loop,1476 - Undefined,243 - Undefined,...,SY,SY,1.0,102775.0,"Interface Americas, Inc.",NaN,NaN,False,False,Interface
356920,205981-000,Interface TP0010 (150010250) Undefined 50cm x ...,INTF,22482 - TP0010,NaN,NaN,118967 - To Be Selected (TBS),2060 - Undefined,825 - 50cm x 50cm,NaN,...,SY,SY,1.0,102775,"Interface Americas, Inc.",NaN,NaN,False,False,Interface
252653,167783-033,Interface Entropy GlasBac (1464802500) Tufted ...,INTF,1977 - Entropy GlasBac,NaN,NaN,82566 - Baltic (7232),2645 - Tufted Tip Shear,494 - 19.69in x 19.69in,NaN,...,SY,SY,1.0,102775.0,"Interface Americas, Inc.",NaN,NaN,False,False,Interface
318335,182409-009,Interface Trio GlasBac (124960AK00) Tufted Cut...,INTF,6297 - Trio GlasBac,NaN,NaN,29502 - Custom (254248),1996 - Tufted Cut & Loop,903 - 25cm x 1m,NaN,...,SY,SY,1.0,102775.0,"Interface Americas, Inc.",NaN,NaN,False,False,Interface
210395,128143-014,Interface Mineral 810 (13824) Tufted Texture L...,INTF,4278 - Mineral 810,NaN,NaN,40435 - Hermatite (4756),2023 - Tufted Texture Loop,494 - 19.69in x 19.69in,NaN,...,SY,SY,1.0,102775,"Interface Americas, Inc.",NaN,NaN,False,False,Interface
127614,167031-005,Interface To Scale w/GlasBac RE (146520250H) T...,INTF,6208 - To Scale w/GlasBac RE,NaN,NaN,87651 - Cross Section (7768),2023 - Tufted Texture Loop,494 - 19.69in x 19.69in,NaN,...,SY,SY,1.0,102775,"Interface Americas, Inc.",NaN,NaN,False,False,Interface
58034,011435,INTERFACE\MICRO LINE\103718 ASH\50CMX50CM,INTF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,CTN,CTN,1.0,999999,SUPPLIER PLACEHOLDER,NaN,NaN,False,False,INTERFACE


In [ ]:
extracted_sample = df_sample['DESCRIPTION'].apply(extract_description_flexible).apply(pd.Series)
df_sample_normalized = pd.concat([df_sample, extracted_sample], axis=1)

display(df_sample_normalized.head())

In [ ]:
df_sample_normalized.to_csv("df_sample_normalized.csv", index=True)

### Create a new naming convention 

In [ ]:
nomenclature_columns = [
    'category', 'collection', 'product_name', 'product_number',
    'backing', 'dimensions', 'name', 'part_number',
    'product_type', 'form', 'color', 'color_code'
]


def create_nomenclature(row):
    values = [str(row[col]) if pd.notnull(row.get(col)) else "N/A" for col in nomenclature_columns]
    return "Interface Inc - " + "-".join(values)



In [46]:
# Apply the nomenclature creation function to each row
df_sample_normalized['new_nomenclature'] = df_sample_normalized.apply(create_nomenclature, axis=1)

In [47]:
df_sample_normalized.to_csv("df_sample_normalized_with_nomenclature.csv", index=True)